In [17]:
# Cell 1: Setup
import sys
sys.path.append('../src')
import numpy as np
from data_loader import DataLoader
from feature_extraction import FeatureExtractor
from model_training import ModelTrainer

# Cell 2: Load Data and Extract Features (reuse from previous)
loader = DataLoader()
X_train, X_val, X_test, y_train, y_val, y_test = loader.prepare_data()

extractor = FeatureExtractor()


Loading MNIST dataset...
✓ Dataset loaded: 70000 samples, 784 features
✓ Pixels normalized using standard
✓ Data split:
   Train = 49000
   Validation = 7000
   Test = 14000


In [18]:
print("\n=== FEATURE EXTRACTION ===")

# ------------------------------------------------
# Extract HOG Features
# ------------------------------------------------
hog_train = extractor.extract_hog_features(X_train)

hog_val = extractor.extract_hog_features(X_val)

hog_test = extractor.extract_hog_features(X_test)
zernike_train = extractor.extract_zernike_moments(X_train)

# ------------------------------------------------
# Apply PCA
# ------------------------------------------------
features_train = extractor.apply_pca(
    hog_train,
    n_components=100,
    fit=True
)

features_val = extractor.apply_pca(
    hog_val,
    fit=False
)

features_test = extractor.apply_pca(
    hog_test,
    fit=False
)

print("\n✓ Feature extraction completed")

print(f"Train Features Shape: {features_train.shape}")
print(f"Validation Features Shape: {features_val.shape}")
print(f"Test Features Shape: {features_test.shape}")




=== FEATURE EXTRACTION ===
Extracting HOG features...
  Processed 5000 images
  Processed 10000 images
  Processed 15000 images
  Processed 20000 images
  Processed 25000 images
  Processed 30000 images
  Processed 35000 images
  Processed 40000 images
  Processed 45000 images
✓ HOG features extracted
  Shape: (49000, 1296)
Extracting HOG features...
  Processed 5000 images
✓ HOG features extracted
  Shape: (7000, 1296)
Extracting HOG features...
  Processed 5000 images
  Processed 10000 images
✓ HOG features extracted
  Shape: (14000, 1296)
Extracting Zernike moments...
  Processed 5000 images
  Processed 10000 images
  Processed 15000 images
  Processed 20000 images
  Processed 25000 images
  Processed 30000 images
  Processed 35000 images
  Processed 40000 images
  Processed 45000 images
✓ Zernike features extracted
  Shape: (49000, 25)
✓ PCA applied
  Components: 100
  Explained Variance: 0.7528 (75.28%)

✓ Feature extraction completed
Train Features Shape: (49000, 100)
Validation

In [19]:

trainer = ModelTrainer()
models = trainer.train_all_models(features_train, y_train)

print("\n=== HYPERPARAMETER TUNING ===")

best_svm = trainer.hyperparameter_tuning_svm(
    features_train,
    y_train,
    C_values=[0.1, 1, 10, 100]
)

trainer.save_models('../models/')

print("\n=== VALIDATION ACCURACY ===")
for model_name, model in trainer.models.items():
    val_acc = model.score(features_val, y_val)
    print(f"{model_name}: {val_acc:.4f}")


TRAINING ALL MODELS

=== TRAINING SVM ===
✓ SVM trained in 5.55s
  Hyperparameters: C=1.0

=== TRAINING KNN ===
✓ KNN trained in 0.02s
  Hyperparameters: n_neighbors=5

=== TRAINING LOGISTIC REGRESSION ===
✓ Logistic Regression trained in 4.97s
  Hyperparameters: C=1.0, max_iter=1000

=== TRAINING RANDOM FOREST ===
✓ Random Forest trained in 9.25s
  Hyperparameters: n_estimators=50, max_depth=None

✓ All models trained successfully!

=== HYPERPARAMETER TUNING ===

=== SVM HYPERPARAMETER TUNING ===
Fitting 3 folds for each of 4 candidates, totalling 12 fits
✓ Best parameters: {'C': 100}
  Best CV score: 0.9644
✓ SVM saved to ../models/SVM.pkl
✓ KNN saved to ../models/KNN.pkl
✓ LogisticRegression saved to ../models/LogisticRegression.pkl
✓ RandomForest saved to ../models/RandomForest.pkl
✓ SVM_Tuned saved to ../models/SVM_Tuned.pkl

=== VALIDATION ACCURACY ===
SVM: 0.9631
KNN: 0.9789
LogisticRegression: 0.9657
RandomForest: 0.9601
SVM_Tuned: 0.9634


In [20]:
# Cell X: Save Fitted PCA
import pickle
import os

print("=== SAVING FITTED PCA ===")

# Verify PCA is fitted
print(f"PCA fitted: {extractor.pca is not None}")
print(f"PCA n_components: {extractor.pca.n_components_}")
print(f"PCA explained variance ratio: {extractor.pca.explained_variance_ratio_.sum():.4f}")

# Create models directory if it doesn't exist
os.makedirs('../models/', exist_ok=True)

# Save fitted PCA
with open('../models/pca_fitted.pkl', 'wb') as f:
    pickle.dump(extractor.pca, f)

print("✓ Fitted PCA saved to ../models/pca_fitted.pkl")


=== SAVING FITTED PCA ===
PCA fitted: True
PCA n_components: 100
PCA explained variance ratio: 0.7528
✓ Fitted PCA saved to ../models/pca_fitted.pkl
